# FoodBridge SG — Donor Surplus Forecast Model
### IS215 Digital Business – Technologies and Transformation

This notebook implements the **donor-facing surplus forecasting pipeline** for FoodBridge SG.  
It takes historical grocery sales data and predicts how much perishable food each store is likely to have as surplus on any given day — enabling supermarket managers (donors) to proactively reduce over-ordering and schedule donation pickups.

**Dataset:** Corporación Favorita Grocery Sales (UCI-style supermarket data)  
**Model:** Random Forest Regressor  
**Output:** `donor_surplus_forecast.csv` — daily predicted surplus per store, for the donor-facing dashboard

---
## Stage 1: Import Libraries

We import the core libraries needed for this pipeline:

- **pandas / numpy** — data loading, manipulation and numerical operations
- **matplotlib / seaborn** — visualisation for EDA and charts
- **scikit-learn** — machine learning tools (model, train/test split, metrics, encoders)
- **warnings** — suppress non-critical deprecation warnings to keep output clean

> If any library is missing, run `pip install pandas numpy scikit-learn matplotlib seaborn` in your terminal first.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully")

Matplotlib is building the font cache; this may take a moment.


✅ Libraries loaded successfully


---
## Stage 2: Load All CSV Files

We load all 7 source files that make up this dataset.

| File | Contents |
|---|---|
| `training.csv` | Historical daily unit sales per store per item |
| `items.csv` | Item metadata including whether it is perishable |
| `stores.csv` | Store locations, types, and cluster assignments |
| `testing.csv` | Future dates for which we need to generate predictions |
| `transactions.csv` | Daily transaction count (customer footfall) per store |
| `oil.csv` | Daily oil price — a macroeconomic demand indicator |
| `holidays_events.csv` | National/regional holidays that affect buying patterns |

All `date` columns are converted to `datetime` type immediately.  
This is essential — without it, date arithmetic (rolling averages, holiday lookups) will fail silently or throw type errors.

In [ ]:
print("\n📂 Loading datasets...")

train = pd.read_csv('training.csv')
items = pd.read_csv('items.csv')
stores = pd.read_csv('stores.csv')
test = pd.read_csv('testing.csv')
transactions = pd.read_csv('transactions.csv')
oil = pd.read_csv('oil.csv')
holidays = pd.read_csv('holidays_events.csv')

# Convert date columns to datetime format
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
transactions['date'] = pd.to_datetime(transactions['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])

print(f"  training.csv     → {train.shape[0]:,} rows")
print(f"  items.csv        → {items.shape[0]:,} rows")
print(f"  stores.csv       → {stores.shape[0]:,} rows")
print(f"  testing.csv      → {test.shape[0]:,} rows")
print(f"  transactions.csv → {transactions.shape[0]:,} rows")
print(f"  oil.csv          → {oil.shape[0]:,} rows")
print(f"  holidays.csv     → {holidays.shape[0]:,} rows")
print("✅ All datasets loaded")


📂 Loading datasets...
  training.csv     → 400,000 rows
  items.csv        → 4,100 rows
  stores.csv       → 54 rows
  testing.csv      → 24,999 rows
  transactions.csv → 83,488 rows
  oil.csv          → 1,218 rows
  holidays.csv     → 350 rows
✅ All datasets loaded


---
## Stage 3: Filter for Perishable Items Only

Not all food items go to waste, only **perishable** items (fresh produce, bread, dairy, deli) spoil if unsold.  
The `items.csv` file has a binary `perishable` column: `1 = perishable`, `0 = non-perishable`.

**Why this matters:**  
Including non-perishable items (e.g. canned goods, cleaning products) would dilute the wastage signal and make the model predict wastage for items that cannot go bad — producing meaningless results.

We filter `items` to perishables only, then do an **inner merge** with training data so only perishable item-store-date rows remain in our working dataframe `df`.

In [5]:
perishable_items = items[items['perishable'] == 1][['item_nbr', 'family', 'class']]

print(f"  Total items: {len(items)}")
print(f"  Perishable items: {len(perishable_items)}")
print(f"  Item families: {perishable_items['family'].unique()}")

# Merge train data with perishable items only
df = train.merge(perishable_items, on='item_nbr', how='inner')

print(f"\n  Training rows after filtering to perishables: {df.shape[0]:,}")
print("✅ Perishable filter applied")

# Preview the filtered data
df.head()

  Total items: 4100
  Perishable items: 986
  Item families: <StringArray>
[  'BREAD/BAKERY',           'DELI',        'POULTRY',           'EGGS',
          'DAIRY',          'MEATS',        'SEAFOOD', 'PREPARED FOODS',
        'PRODUCE']
Length: 9, dtype: str

  Training rows after filtering to perishables: 87,271
✅ Perishable filter applied


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class
0,0,2013-01-01,25,103665,7.0,NaN,BREAD/BAKERY,2712
1,4,2013-01-01,25,108701,1.0,NaN,DELI,2644
2,25,2013-01-01,25,129635,11.0,NaN,DAIRY,2112
3,26,2013-01-01,25,153239,3.0,NaN,BREAD/BAKERY,2712
4,28,2013-01-01,25,153395,7.0,NaN,BREAD/BAKERY,2704


---
## Stage 4: Engineer the Target Variable — Estimated Wastage

The dataset has no direct "wastage" column, so we must **engineer it** as a proxy.

**The core logic:**
> `Estimated Waste = 7-Day Rolling Average Sales − Actual Sales Today`  
> (clipped to 0 — waste can never be negative)

**Why a 7-day rolling average?**  
A 7-day window smooths out weekly seasonal patterns (e.g. busy Fridays) while still being short enough to adapt to recent demand trends. If a store usually sells 10 units but only sold 3 today, the 7 remaining units are likely unsold perishables that will be wasted.

**Why `.shift(1)`?**  
We shift by 1 day so today's actual sales are not included in "today's expected sales", this avoids **data leakage**, where the model would otherwise already know the answer.

**Why `.clip(lower=0)`?**  
If a store sells more than its average (e.g. during a promotion), the difference is negative, but negative waste has no physical meaning, so we floor it at zero.

In [6]:
# Sort data before computing rolling average — essential for correct groupby ordering
df = df.sort_values(['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)

# 7-day rolling average (using prior days only, not today — avoids leakage)
df['avg_sales_7d'] = (
    df.groupby(['store_nbr', 'item_nbr'])['unit_sales']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

# Estimated wastage = gap between expected and actual (never negative)
df['estimated_waste'] = (df['avg_sales_7d'] - df['unit_sales']).clip(lower=0)

# Remove rows where rolling average could not be computed (first record per item)
df = df.dropna(subset=['avg_sales_7d'])

print(f"  Average estimated waste per row: {df['estimated_waste'].mean():.2f} units")
print(f"  Max estimated waste in one row:  {df['estimated_waste'].max():.2f} units")
print(f"  Rows with non-zero waste: {(df['estimated_waste'] > 0).sum():,}")
print("✅ Wastage target variable created")

df[['store_nbr','item_nbr','date','unit_sales','avg_sales_7d','estimated_waste']].head(10)

  Average estimated waste per row: 3.36 units
  Max estimated waste in one row:  800.11 units
  Rows with non-zero waste: 47,885
✅ Wastage target variable created


,store_nbr,item_nbr,date,unit_sales,avg_sales_7d,estimated_waste
1,1,103665,2013-03-01,3.0,2.000000,0.000000
2,1,103665,2013-04-01,2.0,2.500000,0.500000
3,1,103665,2013-05-01,4.0,2.333333,0.000000
4,1,103665,2013-06-01,2.0,2.750000,0.750000
5,1,103665,2013-07-01,1.0,2.600000,1.600000
6,1,103665,2013-09-01,1.0,2.333333,1.333333
7,1,103665,2013-10-01,6.0,2.142857,0.000000
8,1,103665,2013-11-01,3.0,2.714286,0.000000
10,1,108696,2013-04-01,2.0,3.000000,1.000000
11,1,108696,2013-05-01,2.0,2.500000,0.500000


---
## Stage 5: Feature Engineering — Add External Signals

A model that only sees `avg_sales_7d` would miss important context about *why* waste spikes.  
We enrich the dataset with 6 categories of features:

| Feature group | Features added | Why it matters |
|---|---|---|
| **Date** | day_of_week, month, day_of_month, is_weekend | Wastage is higher on certain days (e.g. Monday after low-traffic weekend) |
| **Holidays** | is_holiday, is_day_after_holiday | Stores over-order before holidays; the day after sees high unsold stock |
| **Oil price** | dcoilwtico | Proxy for economic conditions — lower purchasing power → lower sales → more waste |
| **Footfall** | transactions | Fewer customers = more unsold stock at end of day |
| **Store info** | type, city, cluster | Different store formats and locations have different waste profiles |
| **Item** | family_encoded, onpromotion | Promotions boost sales; item family determines perishability pattern |

**Categorical encoding:**  
Machine learning models require numeric inputs. We use `LabelEncoder` to convert `family`, `city`, and `type` from text labels into integer codes.

**Note on `reindex` in EDA charts (Stage 6):**  
After this merge, `is_holiday` is guaranteed to have both 0 and 1 values — but if data is filtered, one value might be absent. We use `.reindex()` in charts to guard against shape mismatches.

In [ ]:
print("\n🔧 Engineering features...")

# 5a. Date features
df['day_of_week']  = df['date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['month']        = df['date'].dt.month
df['day_of_month'] = df['date'].dt.day
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

# 5b. Holiday flags
# Use .assign() to avoid chained-assignment warning on a filtered slice
national_holidays = holidays[
    (holidays['locale'] == 'National') & (holidays['type'] == 'Holiday')
][['date']].drop_duplicates().assign(is_holiday=1)

# Drop column first so re-running the cell doesn't produce duplicate is_holiday_x / is_holiday_y
df = df.drop(columns=['is_holiday'], errors='ignore')
df = df.merge(national_holidays, on='date', how='left')
df['is_holiday'] = df['is_holiday'].fillna(0).astype(int)

holiday_dates = set(national_holidays['date'])
df['is_day_after_holiday'] = df['date'].apply(
    lambda d: 1 if (d - pd.Timedelta(days=1)) in holiday_dates else 0
)

# 5c. Oil price (forward-fill gaps on weekends/public holidays)
oil = oil.sort_values('date')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill()
df = df.drop(columns=['dcoilwtico'], errors='ignore')
df = df.merge(oil, on='date', how='left')
df['dcoilwtico'] = df['dcoilwtico'].fillna(df['dcoilwtico'].median())

# 5d. Customer footfall
df = df.drop(columns=['transactions'], errors='ignore')
df = df.merge(transactions, on=['date', 'store_nbr'], how='left')
df['transactions'] = df['transactions'].fillna(df['transactions'].median())

# 5e. Store metadata
df = df.drop(columns=['city', 'type', 'cluster'], errors='ignore')
df = df.merge(stores[['store_nbr', 'city', 'type', 'cluster']], on='store_nbr', how='left')

# 5f. Encode categoricals → numeric for ML
le = LabelEncoder()
df['family_encoded'] = le.fit_transform(df['family'].astype(str))
df['city_encoded']   = le.fit_transform(df['city'].astype(str))
df['type_encoded']   = le.fit_transform(df['type'].astype(str))
df['onpromotion']    = df['onpromotion'].fillna(0).astype(int)

print("  Features added:")
print("    ✔ day_of_week, month, day_of_month, is_weekend")
print("    ✔ is_holiday, is_day_after_holiday")
print("    ✔ oil price (dcoilwtico)")
print("    ✔ store transactions (footfall)")
print("    ✔ store type, city, cluster")
print("    ✔ item family, onpromotion")
print("✅ Feature engineering complete — final shape:", df.shape)

---
## Stage 6: Exploratory Data Analysis (EDA)

Before training the model, we visualise the data to understand **where and when waste occurs**.  
These charts serve two purposes:
1. **Business insight** — they form the evidence base for your "rationale for change" presentation slide
2. **Feature validation** — they confirm our engineered features are correlated with wastage

**Chart 1 — Wastage by item family:** Identifies which food categories waste the most, guiding NPO collection priorities.  
**Chart 2 — Holiday impact:** Shows whether holidays cause above-average waste — validating our `is_holiday` feature.  
**Chart 3 — Day of week pattern:** Reveals if certain days consistently have more unsold stock.  
**Chart 4 — Store type comparison:** Shows if supermarket format (A/B/C/D/E) affects waste levels.

> `.reindex()` is used on charts 2 and 3 to ensure all expected categories always exist in the grouped result, preventing a `ValueError: shape mismatch` if any group is empty.

In [ ]:
print("\n📊 Generating EDA charts...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Food Wastage Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Chart 1: Average estimated waste per item family
waste_by_family = df.groupby('family')['estimated_waste'].mean().sort_values(ascending=False)
axes[0, 0].bar(waste_by_family.index, waste_by_family.values, color='tomato')
axes[0, 0].set_title('Average Estimated Waste by Item Family')
axes[0, 0].set_xlabel('Item Family')
axes[0, 0].set_ylabel('Avg Estimated Waste (units)')
axes[0, 0].tick_params(axis='x', rotation=45)

# Chart 2: Holiday vs normal day — reindex guards against missing groups
waste_by_holiday = (
    df.groupby('is_holiday')['estimated_waste']
    .mean()
    .reindex([0, 1], fill_value=0)
)
axes[0, 1].bar(['Normal Day', 'Holiday'], waste_by_holiday.values, color=['steelblue', 'orange'])
axes[0, 1].set_title('Average Wastage: Normal Day vs Holiday')
axes[0, 1].set_ylabel('Avg Estimated Waste (units)')

# Chart 3: Day of week — reindex ensures all 7 days always appear
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
waste_by_dow = (
    df.groupby('day_of_week')['estimated_waste']
    .mean()
    .reindex(range(7), fill_value=0)
)
axes[1, 0].bar(day_names, waste_by_dow.values, color='mediumseagreen')
axes[1, 0].set_title('Average Wastage by Day of Week')
axes[1, 0].set_ylabel('Avg Estimated Waste (units)')

# Chart 4: Store type
waste_by_type = df.groupby('type')['estimated_waste'].mean().sort_values(ascending=False)
axes[1, 1].bar(waste_by_type.index, waste_by_type.values, color='mediumpurple')
axes[1, 1].set_title('Average Wastage by Store Type')
axes[1, 1].set_xlabel('Store Type')
axes[1, 1].set_ylabel('Avg Estimated Waste (units)')

plt.tight_layout()
plt.savefig('eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("  📁 Saved: eda_charts.png")
print("✅ EDA complete")

---
## Stage 7: Train the Machine Learning Model

We use a **Random Forest Regressor** to predict `estimated_waste`.

**Why Random Forest?**
- Handles both numerical and encoded categorical features without needing scaling
- Naturally captures non-linear interactions (e.g. "holiday AND high footfall → less waste")
- Provides feature importance rankings out of the box
- Robust to outliers — important since sales data has occasional extreme spikes

**Train/test split (80/20):**  
We hold out 20% of rows for evaluation. `random_state=42` ensures reproducibility.

**Hyperparameters chosen:**
- `n_estimators=100` — 100 decision trees give a stable ensemble without excessive compute time
- `max_depth=10` — limits tree depth to prevent overfitting on the training data
- `n_jobs=-1` — uses all available CPU cores to speed up training

**Evaluation metrics:**
- **MAE (Mean Absolute Error)** — average units the model is off by. Most interpretable for business communication.
- **RMSE (Root Mean Squared Error)** — penalises large errors more heavily. Useful for catching occasional large mis-predictions.

In [ ]:
print("\n🤖 Training the wastage forecasting model...")

FEATURES = [
    'day_of_week', 'month', 'day_of_month', 'is_weekend',
    'is_holiday', 'is_day_after_holiday',
    'onpromotion', 'dcoilwtico', 'transactions',
    'type_encoded', 'cluster', 'city_encoded',
    'family_encoded', 'avg_sales_7d'
]

X = df[FEATURES]
y = df['estimated_waste']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"  Training samples: {X_train.shape[0]:,}")
print(f"  Testing samples:  {X_test.shape[0]:,}")

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n  📈 Model Performance:")
print(f"     Mean Absolute Error (MAE):  {mae:.2f} units")
print(f"     Root Mean Squared Error:    {rmse:.2f} units")
print("  (Lower is better. MAE = average units the prediction is off by.)")
print("✅ Model trained")

---
## Stage 8: Feature Importance — What Drives Wastage?

Random Forest tracks how much each feature reduces prediction error across all trees.  
This gives us a ranked **feature importance** score — the higher the score, the more that feature contributes to accurate wastage predictions.

**Why this matters for the project:**
- Features with high importance are the key levers stores can monitor to reduce waste
- For example, if `avg_sales_7d` ranks highest, it confirms that past demand is the strongest predictor — which validates the rolling average approach
- If `is_holiday` ranks highly, it confirms that stores should adjust orders around holidays

This chart can be used directly in your **technology justification** slide to show the model is grounded in real business logic.

In [ ]:
print("\n🔍 Analysing feature importance...")

importance_df = pd.DataFrame({
    'feature':    FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Feature Importance — What Drives Food Wastage?')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("  📁 Saved: feature_importance.png")
print(f"\n  Top 3 drivers of wastage:")
for _, row in importance_df.head(3).iterrows():
    print(f"     {row['feature']}: {row['importance']:.3f}")

---
## Stage 9: Generate Donor Surplus Forecast

With the trained model, we predict surplus for the **most recent date** in the dataset and aggregate it to the **store level** — answering the key question for each supermarket manager:

> *"How much surplus am I predicted to have tomorrow, and which product category is the biggest contributor?"*

**Aggregation logic:**
- `total_predicted_waste` — sum of predicted surplus across all perishable items at that store
- `top_waste_category` — the item family with the highest predicted surplus (e.g. BREAD/BAKERY)

**Surplus classification for donors:**
| Level | Threshold | Action for Store Manager |
|---|---|---|
| 🔴 HIGH | ≥ 30 units | Reduce tomorrow's order or arrange donation pickup |
| 🟡 MODERATE | 10–29 units | Review ordering quantities |
| 🟢 LOW | < 10 units | Ordering is well-calibrated |

The output is saved as `donor_surplus_forecast.csv` — this file feeds directly into the donor-facing dashboard and can also be consumed by the beneficiary matching engine.

In [ ]:
print("\n🗺️  Generating Donor Surplus Forecast...")

latest_date = df['date'].max()
print(f"  Forecasting for: {latest_date.date()}")

today_df = df[df['date'] == latest_date].copy()

if len(today_df) == 0:
    recent = df[df['date'] >= latest_date - pd.Timedelta(days=7)]
    today_df = recent.groupby(['store_nbr', 'family']).last().reset_index()

today_df['predicted_waste'] = model.predict(today_df[FEATURES])
today_df['predicted_waste'] = today_df['predicted_waste'].clip(lower=0).round(1)

donor_view = (
    today_df.groupby('store_nbr')
    .agg(
        total_predicted_waste=('predicted_waste', 'sum'),
        top_waste_category=('family', lambda x: today_df.loc[x.index]
                            .groupby('family')['predicted_waste']
                            .sum().idxmax())
    )
    .reset_index()
)

donor_view = donor_view.merge(
    stores[['store_nbr', 'city', 'state', 'type']], on='store_nbr', how='left'
)
donor_view = donor_view.sort_values('total_predicted_waste', ascending=False)

def classify_surplus(waste):
    if waste >= 30:   return '🔴 HIGH surplus predicted — consider reducing tomorrow\'s order or arranging donation pickup'
    elif waste >= 10: return '🟡 MODERATE surplus predicted — review ordering quantities'
    else:             return '🟢 LOW surplus predicted — ordering is well-calibrated'

donor_view['surplus_status'] = donor_view['total_predicted_waste'].apply(classify_surplus)
donor_view['forecast_date']  = latest_date.date()

donor_view = donor_view[[
    'forecast_date', 'store_nbr', 'city', 'state', 'type',
    'total_predicted_waste', 'top_waste_category', 'surplus_status'
]]

donor_view.to_csv('donor_surplus_forecast.csv', index=False)
print("  📁 Saved: donor_surplus_forecast.csv")
print("✅ Donor Surplus Forecast generated\n")
donor_view.head(10)

---
## Stage 10: Visualise the Donor Surplus Forecast Dashboard

We produce two final charts to make the surplus forecast visually actionable for store managers:

**Left chart — Top Stores by Predicted Surplus (Donor View):**  
Colour-coded by urgency (red/orange/green), this lets each supermarket's operations team instantly see which stores are predicted to have the most surplus and need to take action — either by reducing tomorrow's orders or by proactively arranging a donation pickup with an NPO.

**Right chart — Predicted Surplus by Item Family:**  
Shows the composition of tomorrow's predicted surplus across all stores — helping procurement teams pinpoint which product categories are consistently over-ordered.

In [ ]:
from matplotlib.patches import Patch

print("\n📊 Plotting Donor Surplus Forecast Dashboard...")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(f'Donor Surplus Forecast Dashboard — {latest_date.date()}',
             fontsize=15, fontweight='bold')

top_stores = donor_view.head(15)
colors = [
    'tomato'          if '🔴' in s else
    'orange'          if '🟡' in s else
    'mediumseagreen'
    for s in top_stores['surplus_status']
]
axes[0].barh(
    top_stores['city'] + ' (Store ' + top_stores['store_nbr'].astype(str) + ')',
    top_stores['total_predicted_waste'],
    color=colors
)
axes[0].set_xlabel('Predicted Surplus (units)')
axes[0].set_title('Top Stores by Predicted Surplus (Donor View)')
axes[0].invert_yaxis()
axes[0].legend(handles=[
    Patch(facecolor='tomato',         label='HIGH — reduce order / arrange pickup'),
    Patch(facecolor='orange',         label='MODERATE — review ordering quantities'),
    Patch(facecolor='mediumseagreen', label='LOW — ordering is well-calibrated')
], loc='lower right', fontsize=8)

waste_by_family_today = (
    today_df.groupby('family')['predicted_waste']
    .sum().sort_values(ascending=False).head(10)
)
axes[1].bar(waste_by_family_today.index, waste_by_family_today.values, color='steelblue')
axes[1].set_title('Predicted Surplus by Item Family (All Stores)')
axes[1].set_xlabel('Item Family')
axes[1].set_ylabel('Total Predicted Surplus (units)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('donor_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("  📁 Saved: donor_dashboard.png")
print("✅ Pipeline complete — donor_surplus_forecast.csv is ready for the donor-facing dashboard")